# Step 4: normalizing the grammar

Steps 1 and 2 insert the glossary expansions in their base (dictionary) form, so the expanded text contains words that are grammatically wrong in their context (e.g. `de 7 annus` instead of `de 7 annis`). This step inflects them, again with the multiple-choice setup (see `normalize.py`):

- Words that were never abbreviated must not change at all. The expansion steps only ever replace abbreviations (tokens ending in a period), so every original word reappears unchanged and in order in the expanded text; aligning the two texts (`inserted_spans`) identifies the words the pipeline inserted. An original word that is indistinguishable from an adjacent identical insertion counts as inserted too, so it can still be normalized. If an original word does *not* reappear, the expanded text was not produced by pure substitution (e.g. by an old full-text-rewrite run) — the vita is then flagged and left untouched.
- Of the inserted words, those that **are a glossary base form** are marked as `[[id|word]]`; words still followed by a period are unexpanded abbreviations and are skipped as well.
- The candidate list for each id is the word's **full paradigm**, generated by `paradigm.py` from the cleaned Wortstamm/Deklination columns and ranked by how often each form actually occurs in the RG corpus. The diocese adjectives from `dioceses.csv` (which have no morphology columns) are added as regular i- and o/a-declension adjectives.
- The model returns JSON mapping each id to the fitting form — or to the current form to keep it (original words that correctly stand in their base form simply stay). Every choice is validated by a membership test against the paradigm, so the model can fix the inflection but can never change the word or corrupt the text.

Every response is recorded with a tier (`changed` / `kept` / `rejected` / `missing`) for auditing.

In [ ]:
from helper_functions import vita_df_to_text, text_to_vita_df, call_chat_ai
from normalize import build_lexicon, diocese_entries, find_lexicon_occurrences, inserted_spans, ranked_forms, build_user_prompt, parse_forms
from expand_rest import Vocabulary
from multiple_choice import apply_choices
import polars as pl
from openai import OpenAI
import json

# API configuration (same setup as in the previous steps)
BASE_URL = "https://chat-ai.academiccloud.de/v1"
MODEL = "gemma-4-31b-it"
api_key = "Key"
MAX_ROW_ATTEMPTS = 5

CLIENT = OpenAI(api_key=api_key, base_url=BASE_URL)

SYSTEM_PROMPT = """**Role:** You are a historian specializing in medieval church history with expert knowledge of the Latin used in the papal registers.

**Task:** You will receive a Latin text in which abbreviations were previously expanded to their dictionary forms, so some words are grammatically wrong in their context. These words are marked as `[[id|word]]`, together with a list of acceptable forms for each id. For every id, select the form that fits the syntax of the surrounding text.

**Instructions:**

1. **Choices:** Choose exactly one form per id, from the given forms only. If the current form already fits the context, return it unchanged. Many marked words were already correct — do not change a word unless its context requires a different case or number.

2. **Context:** Pay attention to prepositions (`de`, `in`, `pro`, `cum`, ...), numerals, agreement with adjacent nouns and adjectives, and genitive attributes. The texts are written in the telegraphic register style of the Repertorium Germanicum: dates, sums of money and archival references are interspersed with the Latin, so judge each word by its local syntactic surroundings.

3. **Occurrences:** The same word can require different forms at different places in the text; judge each occurrence in its own context.

4. **Output format:** Return only a single JSON object mapping every id to the chosen form, e.g. `{"1": "annis", "2": "ecclesia"}`. Do not include explanations, commentary, or any other text.
"""

The lexicon maps every glossary base form to its full paradigm; the corpus vocabulary (words that occur unabbreviated in the full RG) ranks each paradigm by real usage, so the forms the model sees first are the ones that actually occur.

In [ ]:
simple = pl.read_csv("data/simple.csv")
complex = pl.read_csv("data/complex.csv")
dioceses = pl.read_csv("data/dioceses.csv")

entries = pl.concat([
    simple.select("Auflösung", "Wortstamm", "Deklination"),
    complex.select("Auflösung", "Wortstamm", "Deklination"),
]).iter_rows()
LEXICON = build_lexicon(list(entries) + diocese_entries(dioceses.get_column("expansion").to_list()))
print(f"{len(LEXICON)} words in the lexicon")

# the original (unexpanded) texts, both for the vocabulary and for the alignment check
rg_all = pl.read_csv("data/RG_header_sublemma_all.csv").select(["volume", "nr_RG", "nr_suffix", "header_no_tags", "regest_no_tags"])
VOCABULARY = Vocabulary.from_texts(
    rg_all.get_column("header_no_tags").drop_nulls().to_list()
    + rg_all.get_column("regest_no_tags").drop_nulls().to_list()
)
CANDIDATES = ranked_forms(LEXICON, VOCABULARY)

# input: the output of step 3
thrice_expanded = pl.read_csv("data/gemma4/thrice_expanded.csv")

def original_vita_text(volume: int, nr: int) -> str:
    return vita_df_to_text(rg_all.filter((pl.col("volume") == volume) & (pl.col("nr_RG") == nr)))

In [ ]:
def normalize_single(text: str, original_text: str):

    spans = inserted_spans(original_text, text)
    if spans is None:
        # not a pure substitution of the original -> hands off, flag for review
        return {
            "text": text,
            "details": [],
            "errors": [{"message": "Alignment failed: an original word does not reappear in the expanded text"}],
            "user_prompt": None,
            "response": None,
        }

    occurrences = find_lexicon_occurrences(text, LEXICON, spans)
    if not occurrences:
        return None

    user_prompt = build_user_prompt(text, occurrences, CANDIDATES)

    choices, details, errors, dump = None, [], [], None
    for attempt in range(MAX_ROW_ATTEMPTS):
        dump = call_chat_ai(CLIENT, MODEL, SYSTEM_PROMPT, user_prompt)
        content = dump["choices"][0]["message"]["content"]
        choices, details, errors = parse_forms(content, occurrences, CANDIDATES)
        if choices is not None:
            break
    if choices is None:
        # no parseable response after all attempts -> leave everything as it is
        choices = {}

    return {
        "text": apply_choices(text, occurrences, choices),
        "details": details,
        "errors": errors,
        "user_prompt": user_prompt,
        "response": dump,
    }

In [ ]:
volume = 5
nr = 885

vita_df = thrice_expanded.filter((pl.col("volume") == volume) & (pl.col("nr_RG") == nr))
thrice_expanded_text = vita_df_to_text(vita_df)
original_text = original_vita_text(volume, nr)

spans = inserted_spans(original_text, thrice_expanded_text)
occurrences = find_lexicon_occurrences(thrice_expanded_text, LEXICON, spans)

print(thrice_expanded_text)
print('-'*100 + '\n')
for occ in occurrences:
    forms = CANDIDATES[occ.abbreviation]
    shown = ', '.join(forms[:8]) + (', ...' if len(forms) > 8 else '')
    print(f"[[{occ.id}]] {occ.matched}: {shown}")

In [ ]:
result = normalize_single(thrice_expanded_text, original_text)
if result:
    normalized_text = result["text"]
    print(normalized_text)

In [ ]:
for detail in result["details"]:
    if detail["tier"] != "kept":
        print(f"[[{detail['id']}]] {detail['word']} → {detail['form']} ({detail['tier']})")

print()
for error in result["errors"]:
    print(error["message"])

# normalizing a batch

In [ ]:
results = []
normalized = []
testset_ids = thrice_expanded.select("volume", "nr_RG").unique().sort(by="*")

In [ ]:
i = 0

for row in testset_ids.iter_rows(named=True):

    vita_df = thrice_expanded.filter((pl.col("volume") == row["volume"]) & (pl.col("nr_RG") == row["nr_RG"]))
    thrice_expanded_text = vita_df_to_text(vita_df)
    original_text = original_vita_text(row["volume"], row["nr_RG"])

    result = normalize_single(thrice_expanded_text, original_text)
    if result is None:
        print(f"skipped vita #{i} (volume {row['volume']} - nr {row['nr_RG']}) with no inserted glossary base forms")
        normalized.append(vita_df)
        continue

    normalized_text = result["text"]
    normalized.append(text_to_vita_df(normalized_text, row["volume"], row["nr_RG"]))

    results.append({
        "volume": row["volume"],
        "nr_RG": row["nr_RG"],
        "details": result["details"],
        "errors": result["errors"],
        "thrice_expanded_text": thrice_expanded_text,
        "normalized_text": normalized_text,
    })

    i += 1
    if i % 10 == 0:
        print(f"finished processing {i} vitas")

normalized = pl.concat(normalized)

In [ ]:
with open("data/gemma4/results_normalize.json", "w") as file:
    json.dump(results, file, indent=2)

normalized.write_csv("data/gemma4/normalized.csv")

In [ ]:
from collections import Counter

tiers = Counter(detail["tier"] for result in results for detail in result["details"])
for tier, count in tiers.most_common():
    print(f"{tier}: {count}")

print()
changes = Counter(
    f"{detail['word']} → {detail['form']}"
    for result in results for detail in result["details"] if detail["tier"] == "changed"
)
for change, count in changes.most_common(25):
    print(f"{count:4d}  {change}")